# 🎭 Arrays — asking questions

**Time:** ~35 minutes &nbsp;•&nbsp; **Goal:** ask an array a **question** and get an answer back for every one of its 21,238 values at once — then use that to find the moments when something actually happened.

In notebook 1 you described the whole recording. But "the average activity of this neuron" is a boring number. The interesting question is **when** — when the whisker was touched, when the mouse drank, when nothing happened at all.

The recording came with that information. You just haven't used it yet.

## Our data — the trace, plus what the mouse was doing

Run this first.

| array | what it is |
|---|---|
| `dff` | the neuron's activity, one number per frame |
| `cue` | 1 on the frame a whisker cue was delivered |
| `go` | 1 while a **go** cue is on — "there is a reward, go and drink" |
| `nogo` | 1 while a **nogo** cue is on — "no reward, stay put" |
| `reward` | 1 on the frame the mouse actually got a drink |

Every one of these is the **same length** as `dff`, and lines up frame for frame.

In [ ]:
# Run this first. Same neuron as notebook 1, but now we also load the events.
import urllib.request
from pathlib import Path

import numpy as np

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"

NEURON = "neuron_0588_20180828_5.csv"
if not Path(NEURON).exists():
    urllib.request.urlretrieve(DATA_URL + NEURON, NEURON)

# unpack=True hands back one 1-D array per column, in order.
dff, cue, go, nogo, reward = np.loadtxt(
    NEURON, delimiter=",", skiprows=1, usecols=(1, 2, 3, 4, 5), unpack=True
)

FRAME_RATE = 10.8
print("frames:", len(dff))
print("go marker:", go[:20])

## Skill 1 — A comparison asks every value at once

Compare an array to a number and you don't get `True` or `False`. You get **an array of them** — one answer per element:

```python
dff > 1        # array([False, False,  True, ...])
```

That is called a **boolean mask**. Two things make it useful:

- `mask.sum()` **counts** the `True`s, because `True` counts as 1
- `dff[mask]` **selects** just the values where the mask is `True`

In [ ]:
# DEMO - when was this neuron unusually busy?
busy = dff > 1.0

print("the mask itself:", busy[:10])
print("frames above 1.0:", busy.sum(), "out of", len(dff))
print("their values:   ", dff[busy][:8])
print("their average:  ", round(dff[busy].mean(), 3))

### 🎭 DIY challenge

1. Build a mask for frames where the neuron was **below** `-0.5`, and count them.
2. Print the average of those quiet frames.
3. What fraction of the whole recording is above the mean? (`(dff > dff.mean()).mean()` — think about why taking the *mean of a mask* gives you a proportion.)

> 💡 A mask is just an array of `True`/`False`. `.sum()` counts them; `.mean()` gives the fraction.

In [ ]:
# TODO - your turn

## Skill 2 — Masking one array with another

Here is the move that makes this dataset work. `go` is the same length as `dff`, so a mask built from `go` can select from `dff`:

```python
dff[go == 1]     # the neuron's activity, but only while a go cue was on
```

You are using **the mouse's behaviour** to index **the neuron's activity**. Combine masks with `&` (and), `|` (or), `~` (not) — and note the brackets, which are **required**:

```python
dff[(go == 1) & (dff > 0)]
```

In [ ]:
# DEMO - is this neuron interested in the go cue?
during_go = dff[go == 1]
during_nogo = dff[nogo == 1]

print("frames during go:   ", len(during_go))
print("mean during go:     ", round(during_go.mean(), 4))
print("mean during nogo:   ", round(during_nogo.mean(), 4))
print("mean overall:       ", round(dff.mean(), 4))

### 🎭 DIY challenge

Look hard at that demo output. The neuron's average activity during a go cue is **lower** than its average overall.

1. Check the same thing for `reward`. Is it also below average?
2. Count the frames where a go cue was on **and** the neuron was above 1.0: `(go == 1) & (dff > 1.0)`.

Then think about it: does that really mean the neuron does not care about go cues? Or might a neuron take a moment to respond, so that averaging *only the frames the cue was on* misses the answer entirely?

> 💡 Hold that thought. Skill 4 fixes it.

In [ ]:
# TODO - your turn

## Skill 3 — From mask to positions with `np.where`

A mask tells you *which* values. Often you want to know **where** — the frame numbers:

```python
np.where(go == 1)      # (array([43, 44, 45, ...]),)
np.where(go == 1)[0]   # array([43, 44, 45, ...])
```

`np.where` hands back a **tuple** — one array of indices per dimension. Our array is 1-D, so there is one, and `[0]` pulls it out. Once you have indices you can do arithmetic on them: divide by `FRAME_RATE` and they become **times**.

In [ ]:
# DEMO - where, in seconds, was the neuron at its busiest?
loud = np.where(dff > 2.0)[0]

print("frames above 2.0:", loud[:10])
print("as seconds:      ", (loud[:10] / FRAME_RATE).round(1))

# an index array can select, just like a mask
print("their values:    ", dff[loud][:5].round(3))

### 🎭 DIY challenge

1. Use `np.where` to get the frames where `reward == 1`. How many are there?
2. Convert the **first five** of them into minutes.
3. Print the neuron's activity on exactly those five frames.

> 💡 `np.where(...)[0]` gives you an array of indices — and you can use that array inside `dff[...]`.

In [ ]:
# TODO - your turn

## Skill 4 — Finding the *start* of something 🏆

`go` stays at 1 for the whole time the cue is on — about nine frames. So `np.where(go == 1)` gives you 2,286 frames, but the mouse only saw **254 cues**. You don't want every frame of every cue. You want the frame each cue **started**.

`np.diff` gives the change from one element to the next. At the start of a cue it jumps 0 → 1, so the difference is `+1`:

```python
onsets = np.where(np.diff(go) == 1)[0] + 1
```

The `+ 1` is there because `np.diff` returns an array one shorter — `diff[i]` describes the step *into* element `i + 1`.

In [ ]:
# DEMO - 254 cues, not 2286 frames
go_onsets = np.where(np.diff(go) == 1)[0] + 1
nogo_onsets = np.where(np.diff(nogo) == 1)[0] + 1

print("frames with go on:", int(go.sum()))
print("actual go cues:   ", len(go_onsets))
print("actual nogo cues: ", len(nogo_onsets))
print("first few onsets: ", go_onsets[:6])

# now look at the neuron 2 seconds AFTER each cue starts, not during it
after = go_onsets + int(2 * FRAME_RATE)
print("mean 2s after go: ", round(dff[after].mean(), 4))
print("mean overall:     ", round(dff.mean(), 4))

### 🏆 DIY finale — cut out one trial

That last line is the clue. Averaging *during* the cue said the neuron was boring — but two seconds **later** it is running at more than four times its overall average. The neuron does respond. It just takes its time, and the cue is long gone by then.

Calcium is slow: the dye takes about a second to brighten after the cell actually fires. So to see the shape properly you need the whole stretch around each cue, not one frame of it.

1. Take the **first** go onset: `start = go_onsets[0]`.
2. Slice `dff` from 10 frames **before** it to 30 frames **after** it. Print the length — it should be 40.
3. Now do it for every onset, collecting the slices in a list:

```python
windows = []
for start in go_onsets:
    windows.append(dff[start - 10:start + 30])
```

4. Print `len(windows)`, and the length of the first and last one.

> ⚠️ Try the same thing with `nogo_onsets` and check every window is 40 long. The first nogo cue starts at frame 2 — what does `dff[-8:32]` actually give you? That is a real bug, and it is silent.

**Next notebook turns that list of 254 windows into a single 2-D array — and finally answers what this neuron is for.**

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Indexing, masks and fancy indexing, in depth: <https://numpy.org/doc/stable/user/basics.indexing.html>
- `np.where` has a second form — `np.where(cond, a, b)` picks element-wise between two arrays: <https://numpy.org/doc/stable/reference/generated/numpy.where.html>
- [ link — Helfrid to add a boolean-indexing tutorial ]

## Recap

You can now:

- turn a comparison into a **boolean mask**, and `.sum()` it to count
- use one array's mask to select from **another** array — behaviour indexing activity
- combine conditions with `&`, `|`, `~` (brackets required)
- get positions with `np.where`, and turn frames into seconds
- find the **onset** of an event with `np.diff`, instead of every frame of it

You also saw a trap: a mean over the wrong frames gave the wrong answer.

**Next:** `03_arrays_2d_basics` — stacking those 254 windows into a table.